# RecoBridge — Synerise Data Preprocessing

Notebook này chạy pipeline preprocessing out-of-core trên sáu file Synerise Parquet. Logic chính nằm trong package `recobridge_ml` để CLI và notebook luôn dùng cùng một implementation.

Đầu ra gồm cohort, catalog, canonical events, quarantine, schema snapshot, quality report và time-split manifests.

## 1. Chuẩn bị môi trường

Chọn virtual environment của dự án làm Jupyter kernel. Nếu package chưa được cài editable, chạy lệnh sau một lần trong cell mới: `%pip install -e .`

In [1]:
import json
import sys
from pathlib import Path

import duckdb

cwd = Path.cwd().resolve()
if (cwd / 'recobridge_ml').is_dir():
    ML_ROOT = cwd
elif (cwd / 'apps' / 'ml' / 'recobridge_ml').is_dir():
    ML_ROOT = cwd / 'apps' / 'ml'
else:
    raise RuntimeError('Hãy mở notebook từ repo root hoặc thư mục apps/ml')

if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from recobridge_ml.preprocess import PROFILE_SIZES, Settings, run

print(f'ML root: {ML_ROOT}')
print(f'Profiles: {PROFILE_SIZES}')

ML root: E:\Workspace\project\recobridge\apps\ml
Profiles: {'smoke': 1000, 'release': 20000}


## 2. Cấu hình

- `smoke`: 1.000 buyer, phù hợp phát triển.
- `release`: 20.000 buyer, dùng cho artifact bàn giao.
- `FORCE_RERUN=True`: chạy lại và ghi đè artifact của profile đã chọn.

In [2]:
PROFILE = 'smoke'  # đổi thành 'release' cho artifact chính thức
FORCE_RERUN = False
SEED = 42
SESSION_GAP_MINUTES = 30

DATA_DIR = ML_ROOT / 'synerise_dataset'
OUTPUT_DIR = ML_ROOT / 'artifacts' / 'data' / PROFILE

settings = Settings(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    profile=PROFILE,
    seed=SEED,
    session_gap_minutes=SESSION_GAP_MINUTES,
)
settings

Settings(data_dir=WindowsPath('E:/Workspace/project/recobridge/apps/ml/synerise_dataset'), output_dir=WindowsPath('E:/Workspace/project/recobridge/apps/ml/artifacts/data/smoke'), profile='smoke', seed=42, session_gap_minutes=30)

## 3. Chạy preprocessing

Nếu profile đã có `data_manifest.json`, notebook chỉ đọc artifact hiện tại. Đặt `FORCE_RERUN=True` để chạy lại toàn bộ schema scan, checksum và curation. Dữ liệu raw không bị chỉnh sửa.

In [3]:
manifest_path = OUTPUT_DIR / 'data_manifest.json'

if FORCE_RERUN or not manifest_path.exists():
    manifest = run(settings, overwrite=FORCE_RERUN)
    print(f'Pipeline completed: {manifest["data_version"]}')
else:
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(f'Using existing artifact: {manifest["data_version"]}')

print(json.dumps({
    'profile': manifest['profile'],
    'duration_seconds': manifest['duration_seconds'],
    'artifact_count': len(manifest['artifacts']),
}, indent=2, ensure_ascii=False))

Using existing artifact: synerise-smoke-v1
{
  "profile": "smoke",
  "duration_seconds": 141.219,
  "artifact_count": 8
}


## 4. Báo cáo chất lượng và phân chia thời gian

In [4]:
quality = json.loads((OUTPUT_DIR / 'data_quality_report.json').read_text(encoding='utf-8'))
sampling = json.loads((OUTPUT_DIR / 'sampling_manifest.json').read_text(encoding='utf-8'))
split = json.loads((OUTPUT_DIR / 'split_manifest.json').read_text(encoding='utf-8'))

summary = {
    'quality_status': quality['status'],
    'cohort_size': sampling['cohort_size'],
    'catalog_rows': quality['catalog']['valid_unique_rows'],
    'canonical_events': quality['events']['canonical_rows'],
    'duplicates_removed': quality['events']['duplicates_removed'],
    'quarantined_rows': quality['events']['quarantined_rows'],
    'unmatched_item_events': quality['events']['unmatched_item_events'],
    'split_rows': quality['events']['split_rows'],
    'horizon': split['horizon'],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "quality_status": "pass",
  "cohort_size": 1000,
  "catalog_rows": 1534050,
  "canonical_events": 56094,
  "duplicates_removed": 6443,
  "quarantined_rows": 0,
  "unmatched_item_events": 0,
  "split_rows": {
    "test": 6000,
    "train": 43637,
    "validation": 6457
  },
  "horizon": {
    "max": "2022-12-07T23:45:45Z",
    "min": "2022-06-23T04:53:15Z"
  }
}


In [5]:
validation_table = [
    {
        'check': name,
        'passed': result['passed'],
        'details': {key: value for key, value in result.items() if key != 'passed'},
    }
    for name, result in quality['validation_checks'].items()
]
validation_table

[{'check': 'canonical_event_contract',
  'passed': True,
  'details': {'distinct_event_ids': 56094,
   'invalid_split_rows': 0,
   'required_null_rows': 0,
   'row_count': 56094}},
 {'check': 'catalog_contract',
  'passed': True,
  'details': {'distinct_products': 1534050,
   'required_null_rows': 0,
   'row_count': 1534050,
   'vector_length_max': 16,
   'vector_length_min': 16}},
 {'check': 'cohort_size_and_uniqueness',
  'passed': True,
  'details': {'distinct_users': 1000, 'expected': 1000, 'row_count': 1000}},
 {'check': 'events_belong_to_cohort',
  'passed': True,
  'details': {'rows_outside_cohort': 0}},
 {'check': 'session_gap',
  'passed': True,
  'details': {'gap_minutes': 30, 'violations': 0}}]

## 5. Kiểm tra curated Parquet bằng DuckDB

In [6]:
events_path = (OUTPUT_DIR / 'canonical_events.parquet').resolve().as_posix().replace("'", "''")
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

event_distribution = con.execute(f"""
    SELECT split, event_type, count(*) AS row_count
    FROM read_parquet('{events_path}')
    GROUP BY split, event_type
    ORDER BY split, event_type
""").fetchall()

event_distribution

[('test', 'ADD_TO_CART', 397),
 ('test', 'BUY', 194),
 ('test', 'PAGE_VISIT', 4651),
 ('test', 'REMOVE_FROM_CART', 176),
 ('test', 'SEARCH', 582),
 ('train', 'ADD_TO_CART', 2431),
 ('train', 'BUY', 1574),
 ('train', 'PAGE_VISIT', 34394),
 ('train', 'REMOVE_FROM_CART', 1019),
 ('train', 'SEARCH', 4219),
 ('validation', 'ADD_TO_CART', 444),
 ('validation', 'BUY', 246),
 ('validation', 'PAGE_VISIT', 4982),
 ('validation', 'REMOVE_FROM_CART', 170),
 ('validation', 'SEARCH', 615)]

In [7]:
columns = [item[0] for item in con.execute(f"SELECT * FROM read_parquet('{events_path}') LIMIT 5").description]
rows = con.fetchall()
[dict(zip(columns, row)) for row in rows]

[{'event_id': '07334b79ecc4145dc8cdae06c89a56e8',
  'event_time': datetime.datetime(2022, 9, 12, 19, 29, 35, tzinfo=<UTC>),
  'event_type': 'BUY',
  'user_id': '10003271',
  'product_id': 290991,
  'url_id': None,
  'search_vector': None,
  'session_id': '9ffb46064a897d0b2251ff3e5413ed8c',
  'category_id': 302,
  'price_bucket': 58,
  'is_catalog_match': True,
  'split': 'train',
  'source_dataset': 'synerise_recsys_2025',
  'source_partition': 'product_buy.parquet',
  'ingested_at': datetime.datetime(2026, 7, 21, 3, 25, 55, 61582, tzinfo=<UTC>)},
 {'event_id': 'ea192ad1a0c1937dee6fb7dc2ad11739',
  'event_time': datetime.datetime(2022, 8, 11, 20, 4, 30, tzinfo=<UTC>),
  'event_type': 'PAGE_VISIT',
  'user_id': '10022006',
  'product_id': None,
  'url_id': 5333788,
  'search_vector': None,
  'session_id': '31986b0c51e15d67b9798f09f4fc0f9c',
  'category_id': None,
  'price_bucket': None,
  'is_catalog_match': None,
  'split': 'train',
  'source_dataset': 'synerise_recsys_2025',
  'source

## 6. Danh sách artifact

In [8]:
[
    {
        'name': path.name,
        'size_mb': round(path.stat().st_size / (1024 * 1024), 3),
        'path': str(path),
    }
    for path in sorted(OUTPUT_DIR.glob('*'))
    if path.is_file()
]

[{'name': 'canonical_events.parquet',
  'size_mb': 1.586,
  'path': 'E:\\Workspace\\project\\recobridge\\apps\\ml\\artifacts\\data\\smoke\\canonical_events.parquet'},
 {'name': 'catalog.parquet',
  'size_mb': 29.955,
  'path': 'E:\\Workspace\\project\\recobridge\\apps\\ml\\artifacts\\data\\smoke\\catalog.parquet'},
 {'name': 'cohort.parquet',
  'size_mb': 0.003,
  'path': 'E:\\Workspace\\project\\recobridge\\apps\\ml\\artifacts\\data\\smoke\\cohort.parquet'},
 {'name': 'data_manifest.json',
  'size_mb': 0.001,
  'path': 'E:\\Workspace\\project\\recobridge\\apps\\ml\\artifacts\\data\\smoke\\data_manifest.json'},
 {'name': 'data_quality_report.json',
  'size_mb': 0.004,
  'path': 'E:\\Workspace\\project\\recobridge\\apps\\ml\\artifacts\\data\\smoke\\data_quality_report.json'},
 {'name': 'quarantined_events.parquet',
  'size_mb': 0.0,
  'path': 'E:\\Workspace\\project\\recobridge\\apps\\ml\\artifacts\\data\\smoke\\quarantined_events.parquet'},
 {'name': 'sampling_manifest.json',
  'size_m